In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.dates as mdates
from datetime import datetime, timedelta

def extract_flows_from_solution(K, x, E, id0, idend):
    flow_records = []
    for k in K:
        if z[k].solution_value <= 0.5:
            continue
        current_node = id0
        while current_node != idend:
            next_arcs = [e for e in E if e['src']['id'] == current_node and x[e['src']['id'], e['dst']['id'], k].solution_value > 0.5]
            if not next_arcs:
                break
            arc = next_arcs[0]
            src = arc['src']
            dst = arc['dst']
            flow_records.append({
                'Vehicle': k,
                'From': src['loc'],
                'To': dst['loc'],
                'Start_Time': src['time'],
                'End_Time': dst['time'],
                'Flow': 1
            })
            current_node = dst['id']
    return pd.DataFrame(flow_records)

def plot_space_time(df):
    # Convert times to datetime format
    base_time = datetime(2000, 1, 1)
    df['Start_Time_dt'] = df['Start_Time'].apply(lambda m: base_time + timedelta(minutes=m))
    df['End_Time_dt'] = df['End_Time'].apply(lambda m: base_time + timedelta(minutes=m))

    locations = sorted(set(df['From']).union(df['To']))
    loc_pos = {loc: i for i, loc in enumerate(locations)}

    # Plot one figure per vehicle
    for vehicle_id in sorted(df['Vehicle'].unique()):
        df_vehicle = df[df['Vehicle'] == vehicle_id]
        plt.figure(figsize=(12, 6))

        for _, row in df_vehicle.iterrows():
            x = [row['Start_Time_dt'], row['End_Time_dt']]
            y = [loc_pos[row['From']], loc_pos[row['To']]]
            plt.plot(x, y, label=f"Vehicle {vehicle_id + 1}", linewidth=2)

        plt.yticks(list(loc_pos.values()), list(loc_pos.keys()))
        plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

        # x-axis ticks
        min_time = df_vehicle['Start_Time_dt'].min()
        max_time = df_vehicle['End_Time_dt'].max()
        num_xticks = 6
        total_minutes = int((max_time - min_time).total_seconds() / 60)
        tick_interval = max(1, int(total_minutes / num_xticks))
        xticks = [min_time + timedelta(minutes=i) for i in range(0, total_minutes + 1, tick_interval)]
        plt.xticks(xticks)

        plt.xlabel('Time (HH:MM)')
        plt.ylabel('Location')
        plt.title(f'Space-Time Plot for Vehicle {vehicle_id + 1}')
        plt.grid(True)
        plt.tight_layout()
        plt.show()



#these lines to be added before printing the solution
        flow_df = extract_flows_from_solution(K, x, E, id0, idend)
        plot_space_time(flow_df)